# Microsoft Fabric Reference Implementation: Metadata-Driven Lakehouse Pipeline

This notebook demonstrates running the **Accelerator** pipeline within a Microsoft Fabric Spark environment.

> **Prerequisites for Live Fabric Execution**:
> 1. **Workspace & Lakehouse Configuration**:
>    - Create a Microsoft Fabric workspace and a Lakehouse artifact (e.g. `lh_accelerator`).
>    - Attach the Lakehouse as the **Default Lakehouse** to this notebook (using the Lakehouse pane on the left).
> 2. **Code Availability on Spark Cluster**:
>    - Upload the repository `src/accelerator` folder or build a wheel (`python -m build`) and attach it via Workspace Environment / `%pip install`.
>    - Alternatively, sync this repository to OneLake `Files/repo/` and insert it into `sys.path`.
> 3. **Input Files Location**:
>    - Place raw input data into OneLake `Files/raw/olist_customers_dataset.csv` or access via ABFSS path.
>    - For the incremental SCD2 demonstration, a second file `Files/raw/olist_customers_incremental.csv` is provided.
> 4. **Delta Lake Support**:
>    - Pre-installed in all Microsoft Fabric Spark runtimes (Spark 3.4+).

---

## 1. Environment & Path Setup
Ensure `accelerator` is importable on the Spark driver and executor nodes.

In [ ]:
import sys
from pathlib import Path

# If working from cloned OneLake Files:
# sys.path.insert(0, "/lakehouse/default/Files/repo/src")

# Verify accelerator is available
try:
    import accelerator
    from accelerator.adapters.fabric_pipeline import run_fabric_pipeline
    from accelerator.contracts import BatchStatus
    print(f"Accelerator loaded successfully from: {accelerator.__file__}")
except ImportError as exc:
    print(f"Please configure accelerator on the Python path: {exc}")

## 2. Configuration Definition
Configure the pipeline for the Olist Customer dimension with SCD Type 2 history enabled on `customer_zip_code_prefix`, `customer_city`, and `customer_state`.

In [ ]:
from accelerator.contracts import (
    PipelineConfig, SilverConfig, Scd2Config,
    ColumnSchema, FieldType, ExtraColumnsPolicy, MissingColumnsPolicy
)

# Primary Lakehouse paths (configure LAKEHOUSE_NAME to match your workspace)
LAKEHOUSE_NAME = "default"
INPUT_CSV_PATH = f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/Files/raw/olist_customers_dataset.csv"

config = PipelineConfig(
    source_name="olist_customers",
    format="csv",
    input_path=Path(INPUT_CSV_PATH),
    bronze_path=Path("bronze_olist_customers"),
    metadata_store=Path("metadata_batch_control"),
    silver=SilverConfig(
        target_name="dim_customers",
        silver_path=Path("silver_dim_customers"),
        rejected_path=Path("quarantine_dim_customers"),
        primary_keys=["customer_id"],
        extra_columns=ExtraColumnsPolicy.IGNORE,
        missing_columns=MissingColumnsPolicy.NULL,
        columns=[
            ColumnSchema("customer_id", FieldType.STRING, nullable=False),
            ColumnSchema("customer_unique_id", FieldType.STRING, nullable=False),
            ColumnSchema("customer_zip_code_prefix", FieldType.STRING, nullable=True),
            ColumnSchema("customer_city", FieldType.STRING, nullable=True),
            ColumnSchema("customer_state", FieldType.STRING, nullable=True),
        ],
        scd2=Scd2Config(
            enabled=True,
            business_keys=["customer_unique_id"],
            tracking_columns=["customer_zip_code_prefix", "customer_city", "customer_state"],
        )
    )
)
print(f"Pipeline configured for source: {config.source_name} -> target: {config.silver.target_name}")

## 3. Execute Pipeline: Initial Batch Load
Run the pipeline using the active Fabric `spark` session. A deterministic `submission_key` provides idempotency tracking.

In [ ]:
submission_key_1 = "batch-customers-initial-2026"

audit_summary_1 = run_fabric_pipeline(
    config_path=config,
    spark=spark,
    submission_key=submission_key_1,
)

print("=== Initial Execution Summary ===")
print(f"Batch ID:            {audit_summary_1.batch_id}")
print(f"Status:              {audit_summary_1.status.value}")
print(f"Stage:               {audit_summary_1.stage}")
print(f"Records Bronze:      {audit_summary_1.records_bronze}")
print(f"Records Silver:      {audit_summary_1.records_silver}")
print(f"SCD2 Versions:       {audit_summary_1.records_scd2_changed}")
print(f"Reconciliation:      {audit_summary_1.reconciliation_status}")
print(f"Idempotent Replay:   {audit_summary_1.idempotent_replay}")

## 4. Verify Lakehouse Delta Tables
Query the landed Bronze data, Silver SCD2 dimension, and central audit control table.

In [ ]:
# 1. Batch Control & Audit Table
print("--- Metadata Batch Control ---")
spark.sql("SELECT batch_id, status, commit_state, records_persisted, records_silver, reconciliation_status FROM metadata_batch_control ORDER BY batch_start_time DESC LIMIT 5").show(truncate=False)

# 2. Bronze Table Sample
print("--- Bronze Table Sample ---")
spark.sql("SELECT customer_id, customer_city, _ingestion_batch_id, _ingestion_timestamp FROM bronze_olist_customers LIMIT 5").show(truncate=False)

# 3. Silver SCD2 Dimension Table
print("--- Silver SCD2 Dimension Sample ---")
spark.sql("SELECT customer_unique_id, customer_city, _effective_from, _effective_to, _is_current, _silver_batch_id FROM silver_dim_customers LIMIT 5").show(truncate=False)

## 5. Verify Idempotent Replay Safety
Re-submitting the exact same submission key should detect the completed run, skip re-insertion, and return `idempotent_replay=True`.

In [ ]:
replay_summary = run_fabric_pipeline(
    config_path=config,
    spark=spark,
    submission_key=submission_key_1,
)

print(f"Replay Batch ID:          {replay_summary.batch_id}")
print(f"Replay Status:            {replay_summary.status.value}")
print(f"Idempotent Replay Flag:   {replay_summary.idempotent_replay}")
assert replay_summary.idempotent_replay is True, "Expected idempotent replay flag to be True!"

## 6. Incremental Load: Demonstrate SCD Type 2 Evolution
Simulate an incremental batch where an existing customer changes their city/state.
The pipeline should:
1. Close the previous version of the record (`_is_current = false`, `_effective_to = timestamp`).
2. Insert the new version (`_is_current = true`, `_effective_to = null`).
3. Preserve historical records and maintain exactly one current record per business key.

In [ ]:
import dataclasses

# Point to incremental source file with updated customer addresses
INCREMENTAL_CSV_PATH = f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com/Files/raw/olist_customers_incremental.csv"
config_incremental = dataclasses.replace(config, input_path=Path(INCREMENTAL_CSV_PATH))

submission_key_2 = "batch-customers-incremental-2026"

audit_summary_2 = run_fabric_pipeline(
    config_path=config_incremental,
    spark=spark,
    submission_key=submission_key_2,
)

print("=== Incremental Execution Summary ===")
print(f"Batch ID:            {audit_summary_2.batch_id}")
print(f"Status:              {audit_summary_2.status.value}")
print(f"SCD2 New Versions:   {audit_summary_2.records_scd2_changed}")

# Query the dimension history for the updated customer to confirm version evolution:
print("--- Updated Customer Version History ---")
spark.sql("""
    SELECT customer_unique_id, customer_city, _effective_from, _effective_to, _is_current, _silver_batch_id
    FROM silver_dim_customers
    ORDER BY customer_unique_id, _effective_from ASC
""").show(truncate=False)